# Notebook 10: BERT multilingüe con pesos por clase y validación cruzada

Corrige la versión 1 del entrenamiento (notebook 08), que no usaba pesos por clase y se evaluó en una sola partición de 232 artículos. Aquí se comparan, con 5 particiones estratificadas (semilla 42) sobre los 1,543 artículos del corpus de Ecuador Chequea: el clasificador trivial, TF-IDF con regresión logística, TF-IDF con SVM y el BERT multilingüe con pesos por clase.

Métricas principales: F1 macro y recall de la clase Verificado (14% del corpus). El accuracy se reporta pero no es la métrica de decisión.

In [1]:
import numpy as np, pandas as pd, torch, time
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH, BATCH_SIZE, EPOCHS, LR, SEED = 256, 16, 4, 2e-5, 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("corpus_ecuador_final.csv")        # el mismo archivo del notebook 08
X = df["contenido"].fillna("").tolist()
y = df["label_binario"].values
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tokenizer(textos, max_length=MAX_LENGTH, padding="max_length",
                             truncation=True, return_tensors="pt")
        self.y = torch.tensor(etiquetas, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

def predecir(model, loader):
    model.eval(); preds = []
    with torch.no_grad():
        for b in loader:
            out = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE))
            preds.extend(out.logits.argmax(dim=1).cpu().numpy())
    return np.array(preds)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados = []
for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
    t0 = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)
    dl_tr = DataLoader(DS([X[i] for i in tr], y[tr]), batch_size=BATCH_SIZE, shuffle=True)
    dl_te = DataLoader(DS([X[i] for i in te], y[te]), batch_size=BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    conteo = np.bincount(y[tr], minlength=2)
    pesos = torch.tensor(len(tr) / (2 * conteo), dtype=torch.float).to(DEVICE)   # pesos por clase
    loss_fn = torch.nn.CrossEntropyLoss(weight=pesos)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl_tr) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

    for ep in range(EPOCHS):
        model.train()
        for b in dl_tr:
            opt.zero_grad()
            out = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE))
            loss_fn(out.logits, b["labels"].to(DEVICE)).backward()
            opt.step(); sch.step()

    pred = predecir(model, dl_te)
    r = dict(fold=fold, acc=accuracy_score(y[te], pred),
             f1_macro=f1_score(y[te], pred, average="macro"),
             rec_ver=recall_score(y[te], pred, pos_label=1))
    resultados.append(r)
    print(f"Fold {fold}: acc={r['acc']:.3f}  F1 macro={r['f1_macro']:.3f}  "
          f"recall Verificado={r['rec_ver']:.3f}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    del model, opt; torch.cuda.empty_cache()

res = pd.DataFrame(resultados)
f = lambda c: f"{res[c].mean():.3f}±{res[c].std(ddof=0):.3f}"
print("\nBERT multilingüe + pesos por clase, 5 particiones:")
print(f"acc={f('acc')}  F1 macro={f('f1_macro')}  recall Verificado={f('rec_ver')}")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

...

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 4: acc=0.873  F1 macro=0.739  recall Verificado=0.545  (4.0 min)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 5: acc=0.776  F1 macro=0.673  recall Verificado=0.750  (4.0 min)

BERT multilingüe + pesos por clase, 5 particiones:
acc=0.843±0.040  F1 macro=0.735±0.043  recall Verificado=0.695±0.082


## Modelo final

Entrena el BERT multilingüe con pesos por clase sobre los 1,543 artículos completos (4 épocas, tasa de aprendizaje 2e-5, semilla 42) y lo exporta como bert_ecuador.zip. Este es el modelo que usa el dashboard. Su desempeño esperado es el de la validación cruzada de arriba; no tiene una evaluación propia porque se entrena con todo el corpus.

In [3]:
import numpy as np, pandas as pd, torch, shutil, json
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH, BATCH_SIZE, EPOCHS, LR, SEED = 256, 16, 4, 2e-5, 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

df = pd.read_csv("corpus_ecuador_final.csv")
X = df["contenido"].fillna("").tolist()
y = df["label_binario"].values
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tokenizer(textos, max_length=MAX_LENGTH, padding="max_length",
                             truncation=True, return_tensors="pt")
        self.y = torch.tensor(etiquetas, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

torch.manual_seed(SEED); np.random.seed(SEED)
loader = DataLoader(DS(X, y), batch_size=BATCH_SIZE, shuffle=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
conteo = np.bincount(y, minlength=2)
pesos = torch.tensor(len(y) / (2 * conteo), dtype=torch.float).to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=pesos)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = len(loader) * EPOCHS
sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

for ep in range(EPOCHS):
    model.train(); perdida = 0
    for b in loader:
        opt.zero_grad()
        out = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE))
        loss = loss_fn(out.logits, b["labels"].to(DEVICE))
        loss.backward(); opt.step(); sch.step()
        perdida += loss.item()
    print(f"Época {ep + 1}/{EPOCHS}  pérdida media: {perdida / len(loader):.4f}", flush=True)

model.save_pretrained("bert_ecuador")
tokenizer.save_pretrained("bert_ecuador")
with open("bert_ecuador/config_entrenamiento.json", "w") as f:
    json.dump({"modelo": MODEL_NAME, "epocas": EPOCHS, "lr": LR, "batch": BATCH_SIZE,
               "max_length": MAX_LENGTH, "pesos_por_clase": pesos.tolist(), "semilla": SEED,
               "articulos": len(y)}, f, indent=2)
shutil.make_archive("bert_ecuador", "zip", ".", "bert_ecuador")
from google.colab import files
files.download("bert_ecuador.zip")

Dispositivo: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Época 1/4  pérdida media: 0.6962
Época 2/4  pérdida media: 0.5905
Época 3/4  pérdida media: 0.4566
Época 4/4  pérdida media: 0.3320


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, recall_score

df = pd.read_csv("corpus_ecuador_final.csv")
X, y = df["contenido"].fillna(""), df["label_binario"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"acc": "accuracy", "f1_macro": "f1_macro",
           "rec_ver": make_scorer(recall_score, pos_label=1)}
tfidf = lambda: TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
modelos = {
    "Siempre la clase mayoritaria": DummyClassifier(strategy="most_frequent"),
    "TF-IDF + Regresión logística": make_pipeline(tfidf(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "TF-IDF + SVM": make_pipeline(tfidf(), LinearSVC(class_weight="balanced")),
}
for nombre, m in modelos.items():
    r = cross_validate(m, X, y, cv=cv, scoring=scoring)
    f = lambda k: f"{r['test_' + k].mean():.3f}±{r['test_' + k].std():.3f}"
    print(f"{nombre:<30} acc={f('acc')}  F1 macro={f('f1_macro')}  recall Verificado={f('rec_ver')}")

Siempre la clase mayoritaria   acc=0.855±0.001  F1 macro=0.461±0.000  recall Verificado=0.000±0.000
TF-IDF + Regresión logística   acc=0.872±0.010  F1 macro=0.720±0.043  recall Verificado=0.484±0.131
TF-IDF + SVM                   acc=0.881±0.010  F1 macro=0.687±0.045  recall Verificado=0.332±0.093


## Resultados (5 particiones, media ± desviación)

| Modelo | Accuracy | F1 macro | Recall de Verificado |
|---|---|---|---|
| Siempre la clase mayoritaria | 0.855 ± 0.001 | 0.461 ± 0.000 | 0.000 ± 0.000 |
| TF-IDF + Regresión logística | 0.872 ± 0.010 | 0.720 ± 0.043 | 0.484 ± 0.131 |
| TF-IDF + SVM | 0.881 ± 0.010 | 0.687 ± 0.045 | 0.332 ± 0.093 |
| BERT multilingüe con pesos por clase | 0.843 ± 0.040 | 0.735 ± 0.043 | 0.695 ± 0.082 |

El BERT con pesos por clase y la regresión logística quedan empatados en F1 macro: la diferencia (0.015) es menor que la variación entre particiones (cerca de 0.04). El BERT detecta más contenido Verificado (recall 0.70 contra 0.48), a costa de un accuracy menor, que queda por debajo del clasificador trivial (0.855).